In [4]:
"""
STAGE 2: Chunk the cached extracted text and embed it into ChromaDB.

Reads only from extracted_text/*.json (produced by extract_text.py).
Never touches the original PDFs. Safe to re-run any time you want to:
  - change CHUNK_SIZE / CHUNK_OVERLAP
  - switch embedding models
  - rebuild the DB from scratch (just delete ./chroma_db first)

This is the script you'll re-run often while tuning your RAG pipeline.
"""

import os
import json
import chromadb
from sentence_transformers import SentenceTransformer

EXTRACTED_DIR = "/content/drive/MyDrive/startup/extracted_text"
CHUNK_SIZE = 400
CHUNK_OVERLAP = 50
COLLECTION_NAME = "exam_prep"

embedder = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="/content/drive/MyDrive/startup/chroma_db")
collection = client.get_or_create_collection(name=COLLECTION_NAME)


def chunk_text(text, chunk_size, overlap):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks


def main():
    chunk_id = 0
    json_count = 0

    for dirpath, _, filenames in os.walk(EXTRACTED_DIR):
        for fname in sorted(filenames):
            if not fname.lower().endswith(".json"):
                continue

            json_path = os.path.join(dirpath, fname)
            with open(json_path, "r", encoding="utf-8") as f:
                record = json.load(f)

            json_count += 1
            print(f"Embedding: {json_path}  "
                  f"[grade={record['grade']}, subject={record['subject']}, "
                  f"type={record['doc_type']}, chapter={record['chapter']}]")

            # Batch embeddings per-document for speed
            texts_to_embed = []
            metadatas = []

            for page_entry in record["pages"]:
                page_num = page_entry["page"]
                page_text = page_entry["text"]

                for chunk in chunk_text(page_text, CHUNK_SIZE, CHUNK_OVERLAP):
                    texts_to_embed.append(chunk)
                    metadatas.append({
                        "grade": record["grade"],
                        "subject": record["subject"],
                        "doc_type": record["doc_type"],
                        "chapter": record["chapter"] or "",
                        "book": record["book"],
                        "page": page_num,
                    })

            if not texts_to_embed:
                continue

            # Embed all chunks for this document in one batch call (much faster)
            embeddings = embedder.encode(texts_to_embed, show_progress_bar=False).tolist()

            ids = [f"chunk_{chunk_id + i}" for i in range(len(texts_to_embed))]
            collection.add(
                ids=ids,
                embeddings=embeddings,
                documents=texts_to_embed,
                metadatas=metadatas,
            )
            chunk_id += len(texts_to_embed)

    print(f"\nDone. Processed {json_count} cached documents, {chunk_id} total chunks embedded.")


if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding: /content/drive/MyDrive/startup/extracted_text/11/chemistry/chemistry_pt1/kech101.json  [grade=11, subject=Chemistry, type=chapter, chapter=01]
Embedding: /content/drive/MyDrive/startup/extracted_text/11/chemistry/chemistry_pt1/kech102.json  [grade=11, subject=Chemistry, type=chapter, chapter=02]
Embedding: /content/drive/MyDrive/startup/extracted_text/11/chemistry/chemistry_pt1/kech103.json  [grade=11, subject=Chemistry, type=chapter, chapter=03]
Embedding: /content/drive/MyDrive/startup/extracted_text/11/chemistry/chemistry_pt1/kech104.json  [grade=11, subject=Chemistry, type=chapter, chapter=04]
Embedding: /content/drive/MyDrive/startup/extracted_text/11/chemistry/chemistry_pt1/kech105.json  [grade=11, subject=Chemistry, type=chapter, chapter=05]
Embedding: /content/drive/MyDrive/startup/extracted_text/11/chemistry/chemistry_pt1/kech106.json  [grade=11, subject=Chemistry, type=chapter, chapter=06]
Embedding: /content/drive/MyDrive/startup/extracted_text/11/chemistry/chemis

In [5]:
!pip install chromadb
!pip install -q transformers accelerate bitsandbytes flask pyngrok sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 8.7 MB/s eta 0:00:00


In [1]:
# ============================================================
# COLAB CELL 1 — Install dependencies
# ============================================================
!pip install -q transformers accelerate bitsandbytes flask sentencepiece

# Download the cloudflared binary (one-time per Colab session)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00


In [2]:

# ============================================================
# COLAB CELL 2 — Load the model (Mistral 7B Instruct, 4-bit quantized)
# ============================================================
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model (this takes a few minutes on first run)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
print("Model loaded.")


Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded.


In [4]:
def generate_answer(system_prompt: str, user_message: str, max_new_tokens: int = 600) -> str:
    messages = [
        {"role": "user", "content": f"{system_prompt}\n\n{user_message}"}
        # Mistral-Instruct's chat template doesn't support a separate "system"
        # role the way Llama/Claude do, so we fold it into the user turn.
    ]

    # Build the prompt as a STRING first (tokenize=False), then tokenize it
    # ourselves. This avoids version-dependent inconsistencies in what
    # apply_chat_template(return_tensors=...) hands back across transformers
    # versions (sometimes a tensor, sometimes a BatchEncoding dict).
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]
    prompt_length = input_ids.shape[-1]

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response_text = tokenizer.decode(
        output[0][prompt_length:], skip_special_tokens=True
    )
    return response_text.strip()


# Quick sanity test before exposing it to the network
test_answer = generate_answer(
    "You are a helpful tutor. Answer only from the given context.",
    "Context: Photosynthesis occurs in chloroplasts.\n\nQuestion: Where does photosynthesis occur?"
)
print("Test answer:", test_answer)



/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Test answer: Photosynthesis occurs in chloroplasts.


In [5]:

# ============================================================
# COLAB CELL 4 — Start Flask server
# ============================================================
from flask import Flask, request, jsonify
import threading

app = Flask(__name__)

@app.route("/generate", methods=["POST"])
def generate():
    data = request.get_json(force=True)
    system_prompt = data.get("system_prompt", "")
    user_message = data.get("user_message", "")

    if not user_message:
        return jsonify({"error": "user_message is required"}), 400

    try:
        answer = generate_answer(system_prompt, user_message)
        return jsonify({"response": answer})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})

# Run Flask in a background thread so this cell doesn't block the notebook
threading.Thread(target=lambda: app.run(port=5000)).start()
print("Flask server starting on port 5000...")



Flask server starting on port 5000...


In [6]:


# ============================================================
# COLAB CELL 5 — Start Cloudflare Tunnel and grab the public URL
# ============================================================
import subprocess
import re
import time

# Start cloudflared pointed at our local Flask port, capture its output
process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None
print("Waiting for Cloudflare Tunnel to start...")

# cloudflared prints its assigned URL to stderr/stdout — scan the first
# couple hundred lines of output for it.
for _ in range(200):
    line = process.stdout.readline()
    if not line:
        time.sleep(0.1)
        continue
    match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("=" * 60)
    print(f"Your public endpoint is: {public_url}/generate")
    print("Copy this URL into your local query.py as COLAB_ENDPOINT")
    print("=" * 60)
else:
    print("Could not detect the tunnel URL automatically. "
          "Check the cell output above/below for a line containing "
          "'trycloudflare.com' and copy it manually.")

# Keep reading output in the background so the tunnel process doesn't
# fill its buffer and stall. This does NOT block the rest of the notebook.
def _drain_output(proc):
    for line in proc.stdout:
        pass  # discard further lines; tunnel keeps running regardless

threading.Thread(target=_drain_output, args=(process,), daemon=True).start()



Waiting for Cloudflare Tunnel to start...
Your public endpoint is: https://scientific-rep-giant-trans.trycloudflare.com/generate
Copy this URL into your local query.py as COLAB_ENDPOINT


In [ ]:
# ============================================================
# COLAB CELL 6 — Keep-alive (optional, run in its own cell)
# ============================================================
# Helps keep the session active while actively testing.
# Does not bypass Colab's hard session limits on the free tier.

while True:
    time.sleep(60)
    print("Still alive...", flush=True)

Still alive...
Still alive...
Still alive...
Still alive...
Still alive...
